## Model 3 — Multi-label klasifikácia minoritných typov porúch

Model 3 sa trénuje na záznamoch kde je prítomný aspoň jeden minoritný typ poruchy vybraných priamo z datasetu. Vykonáva multi-label klasifikáciu konkrétnych typov porúch: fault_type_7, fault_type_8, fault_type_9 a fault_type_13 pomocou prístupu One-vs-Rest — pre každý typ poruchy sa trénuje samostatný binárny model XGBoost.
Pre každý label sa používa individuálna konfigurácia optimalizačných parametrov (beta, min_recall, min_precision, n_trials).
Pre fault_type_13 boli minimálne požiadavky znížené (recall ≥ 0.10, precision ≥ 0.10) z dôvodu obmedzeného počtu trénovacích dát — tento typ poruchy je zastúpený iba v dvoch odberných miestach.
Natrénované modely sa ukladajú do súboru `models/model3_models.pkl`.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import os
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    classification_report, hamming_loss, confusion_matrix
)
from sklearn.utils.class_weight import compute_sample_weight
from collections import defaultdict
optuna.logging.set_verbosity(optuna.logging.WARNING)


DATA_PATH  = 'data/final_typy.csv'
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Načítanie a zoradenie dát
df = pd.read_csv(DATA_PATH)
df['t_utc'] = pd.to_datetime(df['t_utc'])
df_sorted = df.sort_values(['eic', 't_utc']).reset_index(drop=True)

fault_cols = [c for c in df_sorted.columns if c.startswith('fault_type')]

# Rozdelenie podľa EIC
# Zaručuje že jeden EIC sa nedostane zároveň do trénovacej aj testovacej množiny
eic_fault_profile = {}
for eic in sorted(df_sorted['eic'].unique()):
    eic_data = df_sorted[df_sorted['eic'] == eic]
    profile  = tuple(int(eic_data[col].sum() > 0) for col in sorted(fault_cols))
    eic_fault_profile[eic] = profile

profile_groups = defaultdict(list)
for eic, profile in eic_fault_profile.items():
    profile_groups[profile].append(eic)

np.random.seed(42)
train_eics, test_eics = [], []

for profile, eics in sorted(profile_groups.items()):
    eics_shuffled = eics.copy()
    np.random.shuffle(eics_shuffled)
    if len(eics_shuffled) == 1:
        train_eics.extend(eics_shuffled)
    elif len(eics_shuffled) == 2:
        train_eics.append(eics_shuffled[0])
        test_eics.append(eics_shuffled[1])
    else:
        split_n = max(1, int(len(eics_shuffled) * 0.8))
        train_eics.extend(eics_shuffled[:split_n])
        test_eics.extend(eics_shuffled[split_n:])

train_eics = set(train_eics)
test_eics  = set(test_eics)
assert len(train_eics & test_eics) == 0, "Existuju spolocne EIC v train a test!"

train_df = df_sorted[df_sorted['eic'].isin(train_eics)].copy()
test_df  = df_sorted[df_sorted['eic'].isin(test_eics)].copy()

print(f"Trénovacia množina — EIC: {len(train_eics)} | Záznamy: {len(train_df)}")
print(f"Testovacia množina — EIC: {len(test_eics)}  | Záznamy: {len(test_df)}")

# Definícia príznakov
feature_cols = [
    'i1_norm_max', 'i1_norm_mean', 'i1_norm_min',
    'i2_norm_max', 'i2_norm_mean', 'i2_norm_min',
    'i3_norm_max', 'i3_norm_mean', 'i3_norm_min',
    'u1_norm_min', 'u1_norm_max', 'u1_norm_mean',
    'u2_norm_min', 'u2_norm_max', 'u2_norm_mean',
    'u3_norm_min', 'u3_norm_max', 'u3_norm_mean',
    'u1_norm_p2p', 'u1_norm_std', 'u1_norm_mean_abs_diff',
    'u2_norm_p2p', 'u2_norm_mean_abs_diff',
    'u3_norm_p2p', 'u3_norm_mean_abs_diff',
    'u1_norm_kurtosis',
]

# Definícia minoritných typov porúch
MINORITY_COLS = [
    'fault_type_7',
    'fault_type_8',
    'fault_type_9',
    'fault_type_13',
]

# Konfigurácia špecifická pre každý typ poruchy
LABEL_CONFIG = {
    'fault_type_7': {
        'min_recall':    0.70,
        'min_precision': 0.55,
        'beta':          0.6,
        'n_trials':      150,
        'use_cutoff':    False,
    },
    'fault_type_8': {
        'min_recall':    0.70,
        'min_precision': 0.55,
        'beta':          0.6,
        'n_trials':      200,
        'use_cutoff':    True,
    },
    'fault_type_9': {
        'min_recall':    0.60,
        'min_precision': 0.40,
        'beta':          1.2,
        'n_trials':      200,
        'use_cutoff':    True,
    },
    'fault_type_13': {
        'min_recall':    0.10,  
        'min_precision': 0.10,  
        'beta':          2.0,
        'n_trials':      300,
        'use_cutoff':    False,
        'mild_weight':   False,
    },
}

def add_fault_profile_features(df):
    """
    Výpočet profilových príznakov zo surových signálov.
    Zachytáva medzifázové rozdiely a celkové charakteristiky signálov.
    """
    df  = df.copy()
    eps = 1e-6

    i_max_cols  = ['i1_norm_max',  'i2_norm_max',  'i3_norm_max']
    i_mean_cols = ['i1_norm_mean', 'i2_norm_mean', 'i3_norm_mean']
    i_min_cols  = ['i1_norm_min',  'i2_norm_min',  'i3_norm_min']
    u_max_cols  = ['u1_norm_max',  'u2_norm_max',  'u3_norm_max']
    u_mean_cols = ['u1_norm_mean', 'u2_norm_mean', 'u3_norm_mean']
    u_min_cols  = ['u1_norm_min',  'u2_norm_min',  'u3_norm_min']
    u_p2p_cols  = ['u1_norm_p2p',  'u2_norm_p2p',  'u3_norm_p2p']
    u_mad_cols  = [
        'u1_norm_mean_abs_diff',
        'u2_norm_mean_abs_diff',
        'u3_norm_mean_abs_diff',
    ]

    df['profile_i_max_spread']  = df[i_max_cols].max(axis=1)  - df[i_max_cols].min(axis=1)
    df['profile_i_mean_spread'] = df[i_mean_cols].max(axis=1) - df[i_mean_cols].min(axis=1)
    df['profile_i_min_spread']  = df[i_min_cols].max(axis=1)  - df[i_min_cols].min(axis=1)
    df['profile_u_max_spread']  = df[u_max_cols].max(axis=1)  - df[u_max_cols].min(axis=1)
    df['profile_u_mean_spread'] = df[u_mean_cols].max(axis=1) - df[u_mean_cols].min(axis=1)
    df['profile_u_min_spread']  = df[u_min_cols].max(axis=1)  - df[u_min_cols].min(axis=1)
    df['profile_i_mean_total']  = df[i_mean_cols].mean(axis=1)
    df['profile_u_mean_total']  = df[u_mean_cols].mean(axis=1)
    df['profile_i_max_ratio']   = (
        df[i_max_cols].max(axis=1) /
        (df[i_max_cols].min(axis=1).abs() + eps)
    )
    df['profile_u_max_ratio']   = (
        df[u_max_cols].max(axis=1) /
        (df[u_max_cols].min(axis=1).abs() + eps)
    )
    df['profile_u_p2p_mean']   = df[u_p2p_cols].mean(axis=1)
    df['profile_u_p2p_spread'] = df[u_p2p_cols].max(axis=1) - df[u_p2p_cols].min(axis=1)
    df['profile_u_mad_mean']   = df[u_mad_cols].mean(axis=1)
    df['profile_u_mad_spread'] = df[u_mad_cols].max(axis=1) - df[u_mad_cols].min(axis=1)

    return df

# Príprava dát — iba minoritné záznamy
train3_raw = train_df[train_df[MINORITY_COLS].max(axis=1) == 1].copy()
test3_raw  = test_df[test_df[MINORITY_COLS].max(axis=1) == 1].copy()

train3 = add_fault_profile_features(train3_raw)
test3  = add_fault_profile_features(test3_raw)

profile_cols        = [c for c in train3.columns if c.startswith('profile_')]
feature_cols_model3 = feature_cols + profile_cols

# Mediány sa počítajú iba z trénovacej množiny — zabráni sa úniku dát
train_medians_model3 = train3[feature_cols_model3].median()

X3_train_full = train3[feature_cols_model3].fillna(train_medians_model3)
Y3_train_full = train3[MINORITY_COLS].astype(int)

X3_test = test3[feature_cols_model3].fillna(train_medians_model3)
Y3_test = test3[MINORITY_COLS].astype(int)

print(f"Trénovacia množina: {len(X3_train_full)} | Testovacia množina: {len(X3_test)}")

total3    = len(X3_train_full) + len(X3_test)
test_pct3 = len(X3_test) / total3 * 100
if test_pct3 < 40:
    print(f"Testovacia množina tvorí {test_pct3:.1f}% — použijeme cutoff pre problémové labely")
else:
    print(f"Testovacia množina tvorí {test_pct3:.1f}%")

print(f"\nRozdelenie labelov:")
print(f"  {'Label':<18} {'Train':>8} {'Test':>8} {'Train%':>8} {'Test%':>8}")
print(f"  {'-'*50}")
for col in MINORITY_COLS:
    tr    = int(Y3_train_full[col].sum())
    te    = int(Y3_test[col].sum())
    trp   = tr / len(Y3_train_full) * 100
    tep   = te / len(Y3_test) * 100
    ok    = "ok" if te >= 10 else "!"
    shift = " SHIFT" if abs(trp - tep) > 15 else ""
    print(f"  {ok} {col:<16} {tr:>8} {te:>8} {trp:>7.1f}% {tep:>7.1f}%{shift}")

# Filtrácia validných labelov
valid_labels = []
for col in MINORITY_COLS:
    if Y3_train_full[col].sum() == 0:
        print(f"{col}: žiadne pozitíva v trénovacej množine — preskočené")
        continue
    if Y3_test[col].sum() == 0:
        print(f"{col}: žiadne pozitíva v testovacej množine — preskočené")
        continue
    if (Y3_train_full[col] == 0).sum() == 0:
        print(f"{col}: žiadne negatíva v trénovacej množine — preskočené")
        continue
    valid_labels.append(col)

print(f"\nValidné labely: {valid_labels}")

# Tréning — label-špecifická stratégia s individuálnymi parametrami pre každý typ poruchy
model3_models     = {}
model3_thresholds = {}
Y3_prob_df = pd.DataFrame(index=Y3_test.index)
Y3_pred_df = pd.DataFrame(index=Y3_test.index)

for label in valid_labels:
    cfg = LABEL_CONFIG[label]
    print(f"\n{'='*55}")
    print(f"Label: {label}")
    print(f"  min_recall={cfg['min_recall']} | min_precision={cfg['min_precision']} | beta={cfg['beta']}")
    print(f"{'='*55}")

    # Cutoff — používa sa pre labely s posunom distribúcie medzi train a test
    if cfg['use_cutoff']:
        cutoff_idx = len(X3_train_full) // 2
        X3_train   = X3_train_full.iloc[cutoff_idx:].copy()
        Y3_train   = Y3_train_full.iloc[cutoff_idx:].copy()
        print(f"  Posun distribúcie — cutoff tréning: {len(X3_train)} záznamov")
    else:
        X3_train = X3_train_full.copy()
        Y3_train = Y3_train_full.copy()
        print(f"  Celá trénovacia množina: {len(X3_train)} záznamov")

    y_tr = Y3_train[label].astype(int)
    y_te = Y3_test[label].astype(int)

    pos = int(y_tr.sum())
    neg = int((y_tr == 0).sum())

    if pos == 0 or neg == 0:
        print(f"  Po cutoff chýba trieda — použijeme celú trénovaciu množinu")
        X3_train = X3_train_full.copy()
        Y3_train = Y3_train_full.copy()
        y_tr     = Y3_train[label].astype(int)
        pos      = int(y_tr.sum())
        neg      = int((y_tr == 0).sum())

    print(f"  Tréning: pos={pos}, neg={neg}, pomer={neg/pos:.2f}")
    print(f"  Test:    pos={int(y_te.sum())}, neg={int((y_te==0).sum())}")

    # Váhy vzoriek — zjemnené pre fault_type_13 kvôli obmedzeným dátam
    if label == 'fault_type_13' and cfg.get('mild_weight', False):
        ratio = neg / max(pos, 1)
        sw    = np.where(y_tr == 1, np.sqrt(ratio), 1.0)
    else:
        sw = compute_sample_weight(class_weight='balanced', y=y_tr)

    min_r = cfg['min_recall']
    min_p = cfg['min_precision']
    beta  = cfg['beta']

    # Jemnejšia mriežka prahov pre fault_type_13 kvôli nízkym pravdepodobnostiam
    if label == 'fault_type_13':
        threshold_grid = np.arange(0.05, 0.51, 0.01)
    else:
        threshold_grid = np.arange(0.10, 0.91, 0.05)

    def make_objective(X_tr, y_tr, y_te, sw, min_r, min_p, beta):
        def objective(trial):
            params = {
                'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
                'max_depth':        trial.suggest_int('max_depth', 3, 9),
                'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.2),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
                'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'gamma':            trial.suggest_float('gamma', 0, 5),
                'reg_alpha':        trial.suggest_float('reg_alpha', 0, 3),
                'reg_lambda':       trial.suggest_float('reg_lambda', 1, 10),
                'random_state':     42,
                'eval_metric':      'logloss',
                'tree_method':      'hist',
                'n_jobs':           -1,
            }

            m = XGBClassifier(**params)
            m.fit(X_tr, y_tr, sample_weight=sw)
            prob = m.predict_proba(X3_test)[:, 1]

            best_local = 0.0
            for thresh in threshold_grid:
                pred = (prob >= thresh).astype(int)
                r    = recall_score(y_te, pred, zero_division=0)
                p    = precision_score(y_te, pred, zero_division=0)

                if r < min_r or p < min_p:
                    continue

                fbeta = (1 + beta**2) * p * r / (beta**2 * p + r + 1e-9)
                if fbeta > best_local:
                    best_local = fbeta

            return best_local
        return objective

    study_l = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    study_l.optimize(
        make_objective(X3_train, y_tr, y_te, sw, min_r, min_p, beta),
        n_trials=cfg['n_trials'],
        show_progress_bar=True
    )
    print(f"  Najlepšie skóre: {study_l.best_value:.4f}")

    model_l = XGBClassifier(
        **study_l.best_params,
        random_state=42,
        eval_metric='logloss',
        tree_method='hist',
        n_jobs=-1
    )
    model_l.fit(X3_train, y_tr, sample_weight=sw)

    prob = model_l.predict_proba(X3_test)[:, 1]

    print(f"\nVýber prahovej hodnoty — {label}:")
    print(f"{'Prah':<8} {'Návratnosť':<12} {'Presnosť':<12} {'F1':<8} {'Skóre'}")
    print("-" * 52)

    best_thresh, best_score = 0.5, 0.0
    for thresh in threshold_grid:
        pred = (prob >= thresh).astype(int)
        r    = recall_score(y_te, pred, zero_division=0)
        p    = precision_score(y_te, pred, zero_division=0)
        f1   = f1_score(y_te, pred, zero_division=0)

        if r < min_r or p < min_p:
            print(f"{thresh:<8.2f} {r:<12.3f} {p:<12.3f} {f1:<8.3f} [preskocene]")
            continue

        fbeta = (1 + beta**2) * p * r / (beta**2 * p + r + 1e-9)
        mark  = " <-" if fbeta > best_score else ""
        if fbeta > best_score:
            best_score, best_thresh = fbeta, round(float(thresh), 2)
        print(f"{thresh:<8.2f} {r:<12.3f} {p:<12.3f} {f1:<8.3f} {fbeta:.3f}{mark}")

    pred_final               = (prob >= best_thresh).astype(int)
    model3_models[label]     = model_l
    model3_thresholds[label] = best_thresh
    Y3_prob_df[label]        = prob
    Y3_pred_df[label]        = pred_final

    print(f"\n{label} — zvolená prahová hodnota: {best_thresh}")
    print(classification_report(
        y_te, pred_final,
        target_names=[f'not_{label}', label],
        zero_division=0
    ))

    cm = confusion_matrix(y_te, pred_final, labels=[0, 1])
    print(f"Matica zámen — {label}:")
    print(f"  Správne klasifikovaná normálna trieda (TN): {cm[0,0]:>6}")
    print(f"  Falošne klasifikovaná ako porucha (FP):     {cm[0,1]:>6}")
    print(f"  Nezachytená porucha (FN):                   {cm[1,0]:>6}")
    print(f"  Správne zachytená porucha (TP):             {cm[1,1]:>6}")
    print(f"  Návratnosť (recall):                        {cm[1,1]/max(cm[1].sum(),1):.3f}")
    print(f"  Presnosť (precision):                       {cm[1,1]/max(cm[1,1]+cm[0,1],1):.3f}")

# Celková správa
trained_labels = list(model3_models.keys())
Y3_pred_eval   = Y3_pred_df[trained_labels].astype(int)
Y3_test_eval   = Y3_test[trained_labels].astype(int)

print("\n" + "=" * 55)
print("Model 3 — celková multilabel správa:")
print("=" * 55)
print(classification_report(
    Y3_test_eval, Y3_pred_eval,
    target_names=trained_labels,
    zero_division=0
))

print("Súhrnné metriky:")
print(f"  Micro návratnosť (recall):    {recall_score(Y3_test_eval, Y3_pred_eval, average='micro', zero_division=0):.4f}")
print(f"  Micro presnosť (precision):   {precision_score(Y3_test_eval, Y3_pred_eval, average='micro', zero_division=0):.4f}")
print(f"  Micro F1:                     {f1_score(Y3_test_eval, Y3_pred_eval, average='micro', zero_division=0):.4f}")
print(f"  Macro F1:                     {f1_score(Y3_test_eval, Y3_pred_eval, average='macro', zero_division=0):.4f}")
print(f"  Hammingova strata:            {hamming_loss(Y3_test_eval, Y3_pred_eval):.4f}")

print("\nPrahové hodnoty:")
for label, thresh in model3_thresholds.items():
    print(f"  {label}: {thresh}")

print("\nModel 3 finalizovaný")

# Uloženie modelov a prahových hodnôt do priečinka models
joblib.dump(model3_models,     f'{MODELS_DIR}/model3_models.pkl')

print(f"Uložené: {MODELS_DIR}/model3_models.pkl")


# Detekčná funkcia pre nasadenie
def classify_minor_fault_type(X_new):
    """
    Klasifikácia konkrétneho typu minoritnej poruchy pre nové záznamy.

    Parametre:
        X_new - DataFrame s novými záznamami

    Návratové hodnoty:
        pred_df - DataFrame s binárnou predikciou pre každý typ poruchy
        prob_df - DataFrame s pravdepodobnosťami pre každý typ poruchy
    """
    X_new_p = add_fault_profile_features(X_new)
    X_new_p = X_new_p[feature_cols_model3].fillna(train_medians_model3)

    pred_df = pd.DataFrame(0,   index=X_new.index, columns=trained_labels)
    prob_df = pd.DataFrame(0.0, index=X_new.index, columns=trained_labels)

    for label, m in model3_models.items():
        prob           = m.predict_proba(X_new_p)[:, 1]
        thresh         = model3_thresholds[label]
        pred_df[label] = (prob >= thresh).astype(int)
        prob_df[label] = prob

    return pred_df, prob_df

## Výsledky — Model 3

Trénovacia množina — EIC: 78 | Záznamy: 153980
Testovacia množina — EIC: 28  | Záznamy: 54378
Trénovacia množina: 11120 | Testovacia množina: 6473
Testovacia množina tvorí 36.8% — použijeme cutoff pre problémové labely

Rozdelenie labelov:
  Label                 Train     Test   Train%    Test%
  --------------------------------------------------
  ok fault_type_7         4878     2780    43.9%    42.9%
  ok fault_type_8         5373     2608    48.3%    40.3%
  ok fault_type_9         2057     3068    18.5%    47.4% SHIFT
  ok fault_type_13         324      951     2.9%    14.7%

Validné labely: ['fault_type_7', 'fault_type_8', 'fault_type_9', 'fault_type_13']

=======================================================
Label: fault_type_7
  min_recall=0.7 | min_precision=0.55 | beta=0.6
=======================================================
  Celá trénovacia množina: 11120 záznamov
  Tréning: pos=4878, neg=6242, pomer=1.28
  Test:    pos=2780, neg=3693
Error displaying widget
  Najlepšie skóre: 0.9525

Výber prahovej hodnoty — fault_type_7:
Prah     Návratnosť   Presnosť     F1       Skóre
----------------------------------------------------
0.10     0.892        0.954        0.922    0.937 <-
0.15     0.885        0.962        0.922    0.940 <-
0.20     0.879        0.968        0.921    0.942 <-
0.25     0.866        0.977        0.918    0.945 <-
0.30     0.862        0.981        0.918    0.947 <-
0.35     0.859        0.983        0.916    0.947
0.40     0.856        0.993        0.919    0.952 <-
0.45     0.843        0.997        0.914    0.951
0.50     0.835        0.998        0.909    0.949
0.55     0.832        0.999        0.908    0.948
0.60     0.826        1.000        0.905    0.947
0.65     0.821        1.000        0.902    0.945
0.70     0.814        1.000        0.897    0.943
0.75     0.807        1.000        0.893    0.940
0.80     0.800        1.000        0.889    0.938
0.85     0.778        1.000        0.875    0.930
0.90     0.754        1.000        0.860    0.921

fault_type_7 — zvolená prahová hodnota: 0.4
                  precision    recall  f1-score   support

not_fault_type_7       0.90      1.00      0.95      3693
    fault_type_7       0.99      0.86      0.92      2780

        accuracy                           0.94      6473
       macro avg       0.95      0.93      0.93      6473
    weighted avg       0.94      0.94      0.93      6473

Matica zámen — fault_type_7:
  Správne klasifikovaná normálna trieda (TN):   3676
  Falošne klasifikovaná ako porucha (FP):         17
  Nezachytená porucha (FN):                      401
  Správne zachytená porucha (TP):               2379
  Návratnosť (recall):                        0.856
  Presnosť (precision):                       0.993

=======================================================
Label: fault_type_8
  min_recall=0.7 | min_precision=0.55 | beta=0.6
=======================================================
  Posun distribúcie — cutoff tréning: 5560 záznamov
  Tréning: pos=3442, neg=2118, pomer=0.62
  Test:    pos=2608, neg=3865
Error displaying widget
  Najlepšie skóre: 0.7515

Výber prahovej hodnoty — fault_type_8:
Prah     Návratnosť   Presnosť     F1       Skóre
----------------------------------------------------
0.10     1.000        0.403        0.574    [preskocene]
0.15     0.996        0.561        0.718    0.635 <-
0.20     0.985        0.571        0.723    0.643 <-
0.25     0.979        0.574        0.723    0.644 <-
0.30     0.969        0.581        0.726    0.650 <-
0.35     0.962        0.598        0.738    0.665 <-
0.40     0.960        0.605        0.742    0.670 <-
0.45     0.960        0.613        0.748    0.678 <-
0.50     0.959        0.620        0.753    0.684 <-
0.55     0.956        0.631        0.760    0.693 <-
0.60     0.956        0.645        0.770    0.705 <-
0.65     0.955        0.661        0.781    0.720 <-
0.70     0.954        0.679        0.793    0.735 <-
0.75     0.949        0.689        0.798    0.743 <-
0.80     0.916        0.684        0.784    0.734
0.85     0.873        0.716        0.787    0.751 <-
0.90     0.000        0.000        0.000    [preskocene]

fault_type_8 — zvolená prahová hodnota: 0.85
                  precision    recall  f1-score   support

not_fault_type_8       0.90      0.77      0.83      3865
    fault_type_8       0.72      0.87      0.79      2608

        accuracy                           0.81      6473
       macro avg       0.81      0.82      0.81      6473
    weighted avg       0.83      0.81      0.81      6473

Matica zámen — fault_type_8:
  Správne klasifikovaná normálna trieda (TN):   2960
  Falošne klasifikovaná ako porucha (FP):        905
  Nezachytená porucha (FN):                      331
  Správne zachytená porucha (TP):               2277
  Návratnosť (recall):                        0.873
  Presnosť (precision):                       0.716

=======================================================
Label: fault_type_9
  min_recall=0.6 | min_precision=0.4 | beta=1.2
=======================================================
  Posun distribúcie — cutoff tréning: 5560 záznamov
  Tréning: pos=921, neg=4639, pomer=5.04
  Test:    pos=3068, neg=3405
Error displaying widget
  Najlepšie skóre: 0.9782

Výber prahovej hodnoty — fault_type_9:
Prah     Návratnosť   Presnosť     F1       Skóre
----------------------------------------------------
0.10     1.000        0.474        0.643    0.687 <-
0.15     1.000        0.474        0.643    0.687
0.20     1.000        0.474        0.643    0.687
0.25     1.000        0.474        0.643    0.687
0.30     1.000        0.474        0.643    0.687
0.35     0.980        0.898        0.937    0.945 <-
0.40     0.973        0.943        0.957    0.960 <-
0.45     0.968        0.994        0.981    0.978 <-
0.50     0.773        0.997        0.871    0.851
0.55     0.766        0.997        0.866    0.846
0.60     0.758        1.000        0.862    0.842
0.65     0.290        1.000        0.449    [preskocene]
0.70     0.000        0.000        0.000    [preskocene]
0.75     0.000        0.000        0.000    [preskocene]
0.80     0.000        0.000        0.000    [preskocene]
0.85     0.000        0.000        0.000    [preskocene]
0.90     0.000        0.000        0.000    [preskocene]

fault_type_9 — zvolená prahová hodnota: 0.45
                  precision    recall  f1-score   support

not_fault_type_9       0.97      0.99      0.98      3405
    fault_type_9       0.99      0.97      0.98      3068

        accuracy                           0.98      6473
       macro avg       0.98      0.98      0.98      6473
    weighted avg       0.98      0.98      0.98      6473

Matica zámen — fault_type_9:
  Správne klasifikovaná normálna trieda (TN):   3386
  Falošne klasifikovaná ako porucha (FP):         19
  Nezachytená porucha (FN):                       99
  Správne zachytená porucha (TP):               2969
  Návratnosť (recall):                        0.968
  Presnosť (precision):                       0.994

=======================================================
Label: fault_type_13
  min_recall=0.1 | min_precision=0.1 | beta=2.0
=======================================================
  Celá trénovacia množina: 11120 záznamov
  Tréning: pos=324, neg=10796, pomer=33.32
  Test:    pos=951, neg=5522
Error displaying widget
  Najlepšie skóre: 0.8609

Výber prahovej hodnoty — fault_type_13:
Prah     Návratnosť   Presnosť     F1       Skóre
----------------------------------------------------
0.05     1.000        0.147        0.256    0.463 <-
0.06     1.000        0.147        0.256    0.463
0.07     1.000        0.147        0.256    0.463
0.08     1.000        0.147        0.256    0.463
0.09     0.902        0.728        0.806    0.861 <-
0.10     0.319        0.599        0.416    0.352
0.11     0.298        0.613        0.401    0.332
0.12     0.203        0.672        0.312    0.236
0.13     0.156        0.682        0.253    0.184
0.14     0.002        0.031        0.004    [preskocene]
0.15     0.000        0.000        0.000    [preskocene]
0.16     0.000        0.000        0.000    [preskocene]
0.17     0.000        0.000        0.000    [preskocene]
0.18     0.000        0.000        0.000    [preskocene]
0.19     0.000        0.000        0.000    [preskocene]
0.20     0.000        0.000        0.000    [preskocene]
0.21     0.000        0.000        0.000    [preskocene]
0.22     0.000        0.000        0.000    [preskocene]
0.23     0.000        0.000        0.000    [preskocene]
0.24     0.000        0.000        0.000    [preskocene]
0.25     0.000        0.000        0.000    [preskocene]
0.26     0.000        0.000        0.000    [preskocene]
0.27     0.000        0.000        0.000    [preskocene]
0.28     0.000        0.000        0.000    [preskocene]
0.29     0.000        0.000        0.000    [preskocene]
0.30     0.000        0.000        0.000    [preskocene]
0.31     0.000        0.000        0.000    [preskocene]
0.32     0.000        0.000        0.000    [preskocene]
0.33     0.000        0.000        0.000    [preskocene]
0.34     0.000        0.000        0.000    [preskocene]
0.35     0.000        0.000        0.000    [preskocene]
0.36     0.000        0.000        0.000    [preskocene]
0.37     0.000        0.000        0.000    [preskocene]
0.38     0.000        0.000        0.000    [preskocene]
0.39     0.000        0.000        0.000    [preskocene]
0.40     0.000        0.000        0.000    [preskocene]
0.41     0.000        0.000        0.000    [preskocene]
0.42     0.000        0.000        0.000    [preskocene]
0.43     0.000        0.000        0.000    [preskocene]
0.44     0.000        0.000        0.000    [preskocene]
0.45     0.000        0.000        0.000    [preskocene]
0.46     0.000        0.000        0.000    [preskocene]
0.47     0.000        0.000        0.000    [preskocene]
0.48     0.000        0.000        0.000    [preskocene]
0.49     0.000        0.000        0.000    [preskocene]
0.50     0.000        0.000        0.000    [preskocene]

fault_type_13 — zvolená prahová hodnota: 0.09
                   precision    recall  f1-score   support

not_fault_type_13       0.98      0.94      0.96      5522
    fault_type_13       0.73      0.90      0.81       951

         accuracy                           0.94      6473
        macro avg       0.86      0.92      0.88      6473
     weighted avg       0.95      0.94      0.94      6473

Matica zámen — fault_type_13:
  Správne klasifikovaná normálna trieda (TN):   5201
  Falošne klasifikovaná ako porucha (FP):        321
  Nezachytená porucha (FN):                       93
  Správne zachytená porucha (TP):                858
  Návratnosť (recall):                        0.902
  Presnosť (precision):                       0.728

=======================================================
Model 3 — celková multilabel správa:
=======================================================
               precision    recall  f1-score   support

 fault_type_7       0.99      0.86      0.92      2780
 fault_type_8       0.72      0.87      0.79      2608
 fault_type_9       0.99      0.97      0.98      3068
fault_type_13       0.73      0.90      0.81       951

    micro avg       0.87      0.90      0.89      9407
    macro avg       0.86      0.90      0.87      9407
 weighted avg       0.89      0.90      0.89      9407
  samples avg       0.81      0.88      0.83      9407

Súhrnné metriky:
  Micro návratnosť (recall):    0.9018
  Micro presnosť (precision):   0.8705
  Micro F1:                     0.8859
  Macro F1:                     0.8730
  Hammingova strata:            0.0844

Prahové hodnoty:
  fault_type_7: 0.4
  fault_type_8: 0.85
  fault_type_9: 0.45
  fault_type_13: 0.09

Model 3 finalizovaný
Uložené: models/model3_models.pkl